# Ch.2 — Dimensionality Reduction

> **The story.** The oldest of the three algorithms is also the simplest. **Karl Pearson** published "On lines and planes of closest fit to systems of points in space" in _Philosophical Magazine_ in **1901** — a six-page paper that introduced what he called "principal axes", describing how to find the direction along which a cloud of points spreads the most. **Harold Hotelling** rediscovered the same idea independently in **1933**, renamed it "principal components", and connected it firmly to eigendecomposition of the covariance matrix. For sixty years PCA was dimensionality reduction. Then in **2008**, **Laurens van der Maaten and Geoffrey Hinton** published "Visualizing Data using t-SNE" — replacing the Gaussian in the low-dimensional embedding with a heavier-tailed **Student-t** distribution, solving the "crowding problem." In **2018**, **Leland McInnes, John Healy, and James Melville** published **UMAP** — grounded in algebraic topology, 10–100x faster than t-SNE, with a `transform()` method for new data. Our 440 wholesale customers live in 6-dimensional spending space. K-Means in Ch.1 reached silhouette=0.52. Projecting to 3D with UMAP before re-running K-Means pushes silhouette to **0.57** — a concrete improvement traceable to cleaner distance geometry.
>
> **Where you are in the curriculum.** [Ch.1 — Clustering](../ch01_clustering) ran K-Means and DBSCAN in raw 6D space, reaching silhouette=0.52 with k=4 clusters. Every scatter plot drawn there required projecting to 2D first — but we never chose _how_ to project. Here we make that choice deliberately, understand what each projection preserves and sacrifices, and use the best one to further improve clustering. This is the bridge from "finding clusters" to "understanding and showing them."
>
> **Notation in this chapter.** $X \in \mathbb{R}^{N \times d}$ — the data matrix ($N=440$ customers, $d=6$ features); $C = \frac{1}{N}X^\top X$ — the covariance matrix $\in \mathbb{R}^{d \times d}$; $\lambda_i, \mathbf{v}_i$ — eigenvalue/eigenvector pair of $C$ (PCA principal components); $k$ — number of retained components; $Z = X V_k \in \mathbb{R}^{N \times k}$ — projected data; **explained variance ratio** $\text{EVR}_i = \lambda_i / \sum_j \lambda_j$; for **t-SNE**: $p_{ij}$ — high-d neighbour probabilities, $q_{ij}$ — low-d Student-t probabilities, **perplexity** — effective neighbourhood size; for **UMAP**: $n_{\text{neighbors}}$, $\text{min\_dist}$.

---

## 0 · The Challenge

> **The mission**: Build **SegmentAI** — discover actionable customer segments with silhouette >0.5

**What we know so far:**

- Ch.1: K-Means (k=4) found 4 interpretable segments — silhouette = **0.52** (already above 0.5!)
- Ch.1: DBSCAN flagged 12 noise customers (extreme outlier spenders)
- The 4 clusters have business meaning: HoReCa buyers, Retail buyers, Mixed-spend, Bulk buyers
- **Can't show 6D clusters to stakeholders** — scatter plots need 2D
- **Silhouette can go higher** — correlated features add distance noise in 6D

**What's blocking us:**

The marketing director asks: "Show me the 4 customer segments in a picture." Clusters live in 6-dimensional spending space. No scatter plot exists for 6D. In 6D, Euclidean distances inflate because correlated dimensions (Fresh + Delicatessen, Grocery + Detergents) add redundant noise to every distance calculation.

**What this chapter unlocks:**

Dimensionality reduction — compress 6D to 2D/3D while preserving structure. PCA for interpretable axes, t-SNE for cluster topology, UMAP for downstream clustering. Outcome: UMAP 3D → K-Means silhouette **0.52 → 0.57**.

```mermaid
flowchart LR
 A["440 customers\n6D spending\nsilhouette=0.52"] --> B["PCA\n6D→2D\n85% variance"]
 A --> C["t-SNE\n6D→2D\nlocal topology"]
 A --> D["UMAP\n6D→3D\nglobal+local"]
 B --> E["2D scatter\nstakeholder view"]
 C --> F["Cluster topology\n(distances lie!)"]
 D --> G["K-Means on 3D\nsilhouette=0.57"]
 style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint          | Before          | After                     | This Chapter          |
| ------------------- | --------------- | ------------------------- | --------------------- |
| #1 SEGMENTATION     | silhouette=0.52 | silhouette=0.57           | UMAP 3D re-clustering |
| #2 INTERPRETABILITY | Partial         | PCA loadings explain axes | PC1="total spend"     |
| #3 STABILITY        | Not tested      | Still pending             | Ch.3 bootstrap        |
| #4 SCALABILITY      | PCA O(nd²)      | UMAP scales to 100k       | Both confirmed        |
| #5 VALIDATION       | 0.52            | 0.57                      | Higher with UMAP 3D   |


## Core Idea

**PCA:** Find the directions in feature space along which the data spreads the most, then project onto the top $k$ of those directions. The first direction captures the most variance. The second captures the most of what remains, constrained to be perpendicular to the first. The result is a new coordinate system where the axes are ordered by information content.

> **Optional depth:** The principal components are the eigenvectors of the covariance matrix $C = \frac{1}{N}X^\top X$, sorted by decreasing eigenvalue $\lambda_i$. The projection is $Z = X V_k$ where $V_k \in \mathbb{R}^{d \times k}$ contains the top $k$ eigenvectors as columns. Explained variance ratio $\text{EVR}_i = \lambda_i / \sum_j \lambda_j$ measures each component's information share.

**t-SNE:** Preserve the _neighbourhood structure_ of the high-dimensional data in 2D. Two customers who are similar in 6D spending space should appear near each other in the 2D plot. t-SNE converts similarities to probability distributions, then moves 2D points until the low-dimensional distributions match the high-dimensional ones.

> **Optional depth:** t-SNE minimises $\text{KL}(P \| Q) = \sum_{ij} p_{ij} \log(p_{ij}/q_{ij})$ where $p_{ij}$ are high-d Gaussian similarities and $q_{ij}$ are low-d Student-t similarities. The heavy tail of the Student-t solves the crowding problem — it can separate moderately similar points without crushing them together. **Warning:** distances _between_ clusters are not meaningful; only topology is.

**UMAP:** A faster, topology-based approach that preserves both local neighbourhoods and global structure. Critically, UMAP implements `transform()` — you can map new customers into an existing embedding. That makes it viable as a preprocessing step before clustering.

```
Method | Preserves global structure? | New data? | Speed  | Distances meaningful?
-------|----------------------------|-----------|--------|----------------------
PCA   | Yes (linear)               | Yes       | Fast   | Yes
t-SNE | No                         | No        | Slow   | No (topology only)
UMAP  | Mostly                     | Yes       | Medium | Approximately
```


## Visualising What Each Method Does

**PCA:** Rotates the coordinate axes to align with the directions of maximum variance. The first new axis (PC1) points in the direction the data spreads the most. Think of fitting an ellipse to your customer data cloud and reading off its major axes.

```
Original 6D data cloud (schematic — projected to 2D for illustration):

       Fresh
         ↑
    · ·  |  · ·          Data cloud: elongated diagonally.
   ·  ·  |   · ·         The cloud spreads most along the
  ·    · |  ·  ·         diagonal — that's PC1.
---------+-----------→  Grocery
  ·   ·  |  · ·
   ·  ·  |  ·  ·

PC1 ↗  (direction of max variance — "total spend magnitude")
PC2 ↖  (perpendicular to PC1 — "product mix: fresh/frozen vs grocery/detergents")
```

**Dimensionality reduction pipeline for SegmentAI:**

```mermaid
flowchart LR
    A["Raw features\n440 × 6\nspending data"] --> B["Log transform\nnp.log1p\nskew correction"]
    B --> C["StandardScaler\nzero mean,\nunit variance"]
    C --> D["PCA\n6D → 2D\n85% variance"]
    C --> E["UMAP\n6D → 3D\nglobal structure"]
    D --> F["2D scatter\nstakeholder\nvisualisation"]
    E --> G["K-Means\nre-cluster\nsilhouette=0.57"]
    style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


## Running Example — SegmentAI

You have 440 customers and 4 K-Means clusters from Ch.1 — silhouette=0.52 and a table of centroid spending profiles. The marketing director looks at the table and asks: "Can you just show me a picture?" That question is harder than it sounds. The clusters live in 6-dimensional spending space, and there is no obvious choice for which 2 dimensions to plot. Pick Fresh vs Milk and you lose everything Grocery and Detergents tell you. This chapter makes the projection choice deliberate.

Dataset: **UCI Wholesale Customers** — 440 customers, 6 spending features (log-transformed + standardised, same preprocessing as Ch.1). You will project to 2D with each method and colour by the K-Means labels from Ch.1 — the labels are for visual validation only, not for fitting.


In [ ]:
# TODO: Implement this cell
#  (Setup)
#
# Steps:
# 1. Setup
# 2. Compute `IMG` using `Path()`
# 3. Load and preprocess
# 4. Fit the model -- call `log1p()`
# 5. Fit the model -- call `1()`
# 6. Process data
#
# Hint:
#    IMG = Path(???)
#    scaler = StandardScaler(???)
#    df = pd.read_csv(???)
#    X_log = np.log1p(???)

## PCA: Scree Plot and Explained Variance


In [ ]:
# TODO: Implement this cell
#  (PCA full scree)
#
# Steps:
# 1. PCA full scree
# 2. Plot results -- call `subplots()`
# 3. Call `bar()` to produce the result
# 4. Call `plot()` to produce the result
# 5. Plot results -- call `tight_layout()`
# 6. Compute `k90`
#
# Hint:
#    pca_full = PCA(n_components=???)
#    cumevr = evr.cumsum(???)

## PCA: 2D Projection and Loadings


In [ ]:
# TODO: Implement this cell
#  (PCA 2D projection)
#
# Steps:
# 1. PCA 2D projection
# 2. Compute `loadings` using `DataFrame()`
#
# Hint:
#    pca2 = PCA(n_components=???, random_state=???)
#    X_pca = pca2.fit_transform(???)
#    loadings = pd.DataFrame(???)
#    pca2.fit_transform(???)

In [ ]:
# TODO: Implement this cell
#  (PCA 2D scatter, coloured by K-Means labels)
#
# Steps:
# 1. PCA 2D scatter, coloured by K-Means labels
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `scatter()`
# 4. Plot results -- call `arrow()`
# 5. Plot results -- call `tight_layout()`
#
# Hint:
#    axes = plt.subplots(???)

## t-SNE: Perplexity Sweep

t-SNE preserves **local** neighbourhood structure. The `perplexity` parameter controls the effective number of neighbours. We try 10, 30, and 50 to see how cluster appearance changes.

**Distances between clusters in t-SNE are NOT meaningful** — only topology is.


In [ ]:
# TODO: Implement this cell
#  (t-SNE perplexity sweep)
#
# Steps:
# 1. t-SNE perplexity sweep
# 2. Plot results -- call `TSNE()`
# 3. Plot results -- call `legend()`
#
# Hint:
#    tsne = TSNE(n_components=???, perplexity=???)
#    axes = plt.subplots(???)
#    X_tsne = tsne.fit_transform(???)
#    tsne.fit_transform(???)

## UMAP: n_neighbors Sweep

UMAP preserves **both local and global** structure. `n_neighbors` controls the balance.
Unlike t-SNE, UMAP has `transform()` for new data.


In [ ]:
# TODO: Implement this cell
#  (UMAP n_neighbors sweep)
#
# Steps:
# 1. UMAP n_neighbors sweep
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `UMAP()`
# 4. Plot results -- call `legend()`
# 5. Process data
#
# Hint:
#    axes = plt.subplots(???)
#    reducer = umap.UMAP(???)
#    X_umap = reducer.fit_transform(???)
#    reducer.fit_transform(???)

## PCA vs t-SNE vs UMAP: Side-by-Side Comparison


In [ ]:
# TODO: Implement this cell
#  (Side-by-side comparison)
#
# Steps:
# 1. Side-by-side comparison
# 2. Plot results -- call `subplots()`
# 3. Fit the model -- call `UMAP()`
# 4. Plot results -- call `scatter()`
# 5. Plot results -- call `legend()`
#
# Hint:
#    X_pca2 = PCA(n_components=???, random_state=???)
#    X_tsne2 = TSNE(n_components=???, perplexity=???)
#    axes = plt.subplots(???)
#    X_umap2 = umap.UMAP(???)

## Re-Clustering in PCA Space: Does Dimensionality Reduction Help?

Hypothesis: 6D distances are noisy (curse of dimensionality). Clustering in PCA 2D should give tighter segments.


In [ ]:
# TODO: Implement this cell
#  (Re-cluster in PCA 2D)
#
# Steps:
# 1. Re-cluster in PCA 2D
# 2. Fit the model -- call `KMeans()`
# 3. Call `6D()` to produce the result
#
# Hint:
#    km_6d = KMeans(n_clusters=???, init=???)
#    km_pca = KMeans(n_clusters=???, init=???)

## What Can Go Wrong: t-SNE Distance Lie

Distances between clusters in t-SNE are meaningless. This demo shows two groups that are VERY different in original space but appear close in t-SNE, and vice versa.


In [ ]:
# TODO: Implement this cell
#  (t-SNE distance warning)
#
# Steps:
# 1. t-SNE distance warning
# 2. Compute `centroids_tsne` using `array()`
# 3. Call `DataFrame()` to produce the result
#
# Hint:
#    centroids_tsne = np.array(???)

## What Can Go Wrong: PCA Misses Non-Linear Structure


In [ ]:
# TODO: Implement this cell
#  (PCA reconstruction error)
#
# Steps:
# 1. PCA reconstruction error
# 2. Fit the model -- call `PCA()`
# 3. Process data
#
# Hint:
#    pca_k = PCA(n_components=???, random_state=???)
#    X_reduced = pca_k.fit_transform(???)
#    X_reconstructed = pca_k.inverse_transform(???)
#    evr_total = pca_k.explained_variance_ratio_.sum(???)

## Exercises

1. **UMAP min_dist sweep.** Try `min_dist` ∈ {0.0, 0.1, 0.5, 1.0} with `n_neighbors=15`. Plot all four embeddings. How does min_dist affect cluster compactness?

2. **PCA as preprocessing for K-Means.** Run K-Means (K=5) on PCA with 2, 3, 4, and 5 components. Compute silhouette for each. Which number of components gives the best clustering?

3. **t-SNE reproducibility test.** Run t-SNE (perplexity=30) with 5 different `random_state` values. Plot all 5 embeddings. How much does the layout change? Does cluster membership change?


In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above